# Regime-Conditioned Constrained Quantum Optimization - Nifty 50 portfolio selection (final notebook)
Pipeline: leak-free regime inference (classical HMM / quantum VQC) -> regime-blended mean-variance cost -> exact / simulated-annealing / CVaR-QAOA selection of K of N stocks -> walk-forward backtest.
**Run all cells top to bottom** (about 15 minutes on Colab, mostly the Phase 6 library and the Phase 7 VQC training). Sections 1-2 are your original code, unchanged; sections 3-7 are new. Results are saved to `results/` at the end.

## 0. Setup

In [ ]:
!pip install -q "qiskit>=1.2" "hmmlearn>=0.3" yfinance statsmodels
import qiskit, numpy, scipy, pandas, hmmlearn
print({m.__name__: m.__version__ for m in (qiskit, numpy, scipy, pandas, hmmlearn)})   # record these for reproducibility
# Optional: keep the price cache across Colab sessions
# from google.colab import drive; drive.mount('/content/drive'); import os; os.chdir('/content/drive/MyDrive/quantum_portfolio')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import qiskit

# Statistics and Hidden Markov Models
import scipy.stats as stats
import statsmodels.api as sm
from hmmlearn import hmm

print("Libraries successfully imported:")
print(f"- numpy: {np.__version__}")
print(f"- pandas: {pd.__version__}")
print(f"- qiskit: {qiskit.__version__}")

## 1. Phase 1 - Data, frozen universe, features, walk-forward splitter, leakage tests (your code)

In [ ]:
import os
import numpy as np
import pandas as pd
import yfinance as yf

# =====================================================================
# 1. CONFIGURATION & CACHING
# =====================================================================
CACHE_FILE = "nifty50_prices.csv"

# A selection of Nifty 50 constituent tickers (with .NS suffix for Yahoo Finance)
NIFTY50_TICKERS = [
    "RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "ICICIBANK.NS", "INFY.NS",
    "HINDUNILVR.NS", "ITC.NS", "LT.NS", "SBIN.NS", "BHARTIARTL.NS",
    "KOTAKBANK.NS", "AXISBANK.NS", "BAJFINANCE.NS", "M&M.NS", "MARUTI.NS"
]

# Frozen selected universe (11 names) picked based on sector diversity and liquidity
SELECTED_UNIVERSE = [
    "RELIANCE.NS",    # Energy
    "TCS.NS",         # IT
    "HDFCBANK.NS",    # Financials
    "HINDUNILVR.NS",  # FMCG
    "LT.NS",          # Construction / Capital Goods
    "SBIN.NS",        # PSU Bank
    "BHARTIARTL.NS",  # Telecom
    "M&M.NS",         # Auto
    "SUNPHARMA.NS",   # Pharma
    "ULTRACEMCO.NS",  # Materials / Cement
    "POWERGRID.NS"    # Utilities
]

def load_data(start_date="2018-01-01"):
    """
    Downloads historical daily prices or loads them from the local cache.
    """
    if os.path.exists(CACHE_FILE):
        print(f"Loading data from local cache: {CACHE_FILE}")
        df = pd.read_csv(CACHE_FILE, index_col=0, parse_dates=True)
    else:
        print("Downloading Nifty 50 data from Yahoo Finance...")
        # Fetch data for selected universe + some candidates for screening verification
        all_tickers = list(set(NIFTY50_TICKERS + SELECTED_UNIVERSE + ["SUNPHARMA.NS", "ULTRACEMCO.NS", "POWERGRID.NS"]))
        raw_data = yf.download(all_tickers, start=start_date, end=end_date)

        if "Adj Close" in raw_data:
            df = raw_data["Adj Close"]
        else:
            df = raw_data["Close"]

        df.to_csv(CACHE_FILE)
        print(f"Data cached locally to {CACHE_FILE}")
    return df

# =====================================================================
# 2. PRE-SCREENING RULE & FILTERING
# =====================================================================
def screen_universe(df, target_count=11):
    """
    Documents and executes the pre-screening rule:
    1. Filter out tickers with more than 5% missing values.
    2. Select the top diversified tickers representing different sectors.
    """
    missing_ratios = df.isnull().mean()
    valid_tickers = missing_ratios[missing_ratios < 0.05].index.tolist()

    # Keep only those in our predefined frozen list
    final_tickers = [t for t in SELECTED_UNIVERSE if t in valid_tickers]
    return df[final_tickers]

# =====================================================================
# 3. FEATURE ENGINEERING
# =====================================================================
def compute_features(df, vol_window=20, mom_window=20):
    """
    Computes log returns, rolling volatility, and momentum.
    Ensures no future data leakage by strictly using shift() and rolling() parameters.
    """
    features = {}

    for col in df.columns:
        prices = df[col]
        # Log Returns (strictly historical, relying on past 1-day step)
        log_ret = np.log(prices / prices.shift(1))

        # Rolling Volatility (historical backward-looking window)
        # closed='left' parameter prevents including the current day's price in the window calculation
        # but to keep it simple, we compute standard rolling and shift by 1 to make it strictly predictive/historical
        roll_vol = log_ret.rolling(window=vol_window).std().shift(1)

        # Momentum (simple past return over the mom_window, shifted to prevent leak)
        momentum = log_ret.rolling(window=mom_window).sum().shift(1)

        features[f"{col}_log_return"] = log_ret
        features[f"{col}_volatility"] = roll_vol
        features[f"{col}_momentum"] = momentum

    features_df = pd.DataFrame(features, index=df.index)
    return features_df

# =====================================================================
# 4. WALK-FORWARD SPLITTER
# =====================================================================
class WalkForwardSplitter:
    def __init__(self, train_size=252, test_size=63, step_size=63, expanding=False):
        self.train_size = train_size
        self.test_size = test_size
        self.step_size = step_size
        self.expanding = expanding

    def split(self, df):
        """
        Generates (train_indices, test_indices) tuples.
        """
        n_samples = len(df)
        start = 0

        while True:
            train_start = 0 if self.expanding else start
            train_end = start + self.train_size
            test_end = train_end + self.test_size

            if test_end > n_samples:
                break

            yield list(range(train_start, train_end)), list(range(train_end, test_end))
            start += self.step_size

# =====================================================================
# 5. UNIT TESTS & LEAKAGE CHECKS
# =====================================================================
def run_unit_tests():
    print("\n--- Running Unit Tests ---")

    # 1. Load data
    df_raw = load_data()
    df_screened = screen_universe(df_raw)

    # Test 1: Check universe size
    assert len(df_screened.columns) == 11, f"Expected 11 names, got {len(df_screened.columns)}"
    print("Test 1 Passed: Selected exactly 11 tickers.")

    # 2. Compute features
    df_features = compute_features(df_screened)

    # Test 2: Alignment & NaNs check (excluding initial warm-up rows)
    warmup_period = 21
    test_slice = df_features.iloc[warmup_period:]
    assert not test_slice.isnull().any().any(), "Detected unexpected NaNs after warm-up period!"
    print("Test 2 Passed: No NaNs present after the expected warm-up period.")

    # Test 3: Walk-Forward alignment
    splitter = WalkForwardSplitter(train_size=252, test_size=63, step_size=63, expanding=False)
    splits = list(splitter.split(df_screened))
    assert len(splits) > 0, "No splits were generated."

    first_train, first_test = splits[0]
    assert max(first_train) < min(first_test), "Overlapping train and test indices found!"
    print("Test 3 Passed: Splitter ensures strict temporal alignment without overlap.")

    # Test 4: Explicit look-ahead leakage check
    # Modify historical data and check that future features do not change
    original_features = compute_features(df_screened).copy()
    modified_df = df_screened.copy()

    # Perturb a value in the future (e.g. index 100)
    target_idx = 100
    col_to_test = df_screened.columns[0]
    modified_df.iloc[target_idx, modified_df.columns.get_loc(col_to_test)] += 10.0

    modified_features = compute_features(modified_df)

    # Check that any feature before target_idx is absolutely unchanged
    for col in modified_features.columns:
        pd.testing.assert_series_equal(
            original_features[col].iloc[:target_idx],
            modified_features[col].iloc[:target_idx],
            obj=f"Feature {col} before index {target_idx} is affected by future changes (Look-ahead Leakage!)"
        )
    print("Test 4 Passed: No look-ahead leakage detected. Future changes do not affect past features.")
    print("All Phase 1 tests passed successfully!")

# Run the pipeline and unit tests
run_unit_tests()

## 2a. Market regime HMM, full-sample (your code - used for visualisation and as a smoke test only; the backtests below refit walk-forward)

In [ ]:
import numpy as np
import pandas as pd
from hmmlearn import hmm

# =====================================================================
# 1. LOAD DATA & COMPUTE FEATURES FOR ALL SELECTED STOCKS
# =====================================================================
raw_prices = load_data()
screened_prices = screen_universe(raw_prices)
features_df = compute_features(screened_prices)

# =====================================================================
# 2. CREATE A MARKET PROXY & FIT THE 3-STATE MARKET HMM
# =====================================================================
# Calculate market return as the equal-weighted average of the selected constituents' log returns
log_return_cols = [col for col in features_df.columns if col.endswith("_log_return")]
market_log_returns = features_df[log_return_cols].mean(axis=1)

# Combine market return with time index and clean NaNs
market_df = pd.DataFrame({"Market_Log_Return": market_log_returns}).dropna()

# Prepare input for HMM
X_market = market_df[["Market_Log_Return"]].values

# Fit 3-state Gaussian HMM representing overall market conditions
market_hmm = hmm.GaussianHMM(n_components=3, covariance_type="diag", random_state=42, n_iter=100)
market_hmm.fit(X_market)

market_states = market_hmm.predict(X_market)
market_posterior_probs = market_hmm.predict_proba(X_market)

# =====================================================================
# 3. SORT REGIMES BY MARKET VOLATILITY (0 = Low, 1 = Med, 2 = High)
# =====================================================================
state_variances = np.array([np.var(X_market[market_states == i]) for i in range(3)])
sorted_indices = np.argsort(state_variances)
state_map = {old_idx: new_idx for new_idx, old_idx in enumerate(sorted_indices)}

# Apply sorted mapping to market states and posterior probabilities
sorted_market_states = np.vectorize(state_map.get)(market_states)
sorted_market_probs = market_posterior_probs[:, sorted_indices]

# Add market metrics back to a common market dataframe
market_results = pd.DataFrame(index=market_df.index)
market_results["Market_Regime"] = sorted_market_states
market_results["Prob_Market_Regime_0_Low_Vol"] = sorted_market_probs[:, 0]
market_results["Prob_Market_Regime_1_Med_Vol"] = sorted_market_probs[:, 1]
market_results["Prob_Market_Regime_2_High_Vol"] = sorted_market_probs[:, 2]

# =====================================================================
# 4. GENERATE INDIVIDUAL DATA_FRAMES FOR ALL SELECTED STOCKS
# =====================================================================
stock_dfs = {}
for stock in SELECTED_UNIVERSE:
    # Extract individual indicators
    single_stock_df = pd.DataFrame({
        "Price": screened_prices[stock],
        "Log_Return": features_df[f"{stock}_log_return"],
        "Volatility": features_df[f"{stock}_volatility"],
        "Momentum": features_df[f"{stock}_momentum"]
    })

    # Align with the market regime and probabilities
    aligned_stock_df = single_stock_df.join(market_results, how="inner")
    stock_dfs[stock] = aligned_stock_df

# =====================================================================
# 5. DISPLAY SAMPLE RESULTS
# =====================================================================
print("=== MARKET REGIME SUMMARY STATS ===")
for regime_idx in range(3):
    count = (sorted_market_states == regime_idx).sum()
    mean_ret = X_market[sorted_market_states == regime_idx].mean() * 100
    std_ret = X_market[sorted_market_states == regime_idx].std() * 100
    name = ['Low Vol', 'Medium Vol', 'High Vol'][regime_idx]
    print(f"Market Regime {regime_idx} ({name}): Count = {count}, Mean Return = {mean_ret:.4f}%, Volatility = {std_ret:.4f}%")

test_stock = SELECTED_UNIVERSE[0]
print(f"\n=== ALIGNED DATAFRAME SAMPLE FOR {test_stock} ===")
print(stock_dfs[test_stock].head(10).to_string())

In [ ]:
import matplotlib.pyplot as plt

# =====================================================================
# VISUALIZE MARKET REGIMES & STOCK FEATURES
# =====================================================================
test_stock = SELECTED_UNIVERSE[0] # RELIANCE.NS
plot_df = stock_dfs[test_stock].copy()

# Define a color map for the 3 sorted volatility regimes
colors = {0: "green", 1: "orange", 2: "red"}
labels = {0: "Low Volatility", 1: "Medium Volatility", 2: "High Volatility"}

fig, axes = plt.subplots(4, 1, figsize=(15, 12), sharex=True)

# Plot 1: Stock Price colored by Market Regime
axes[0].plot(plot_df.index, plot_df["Price"], color="gray", alpha=0.5, label=f"{test_stock} Price")
for regime in [0, 1, 2]:
    mask = plot_df["Market_Regime"] == regime
    axes[0].scatter(plot_df.index[mask], plot_df["Price"][mask],
                    color=colors[regime], label=labels[regime], s=5, alpha=0.8)
axes[0].set_title(f"{test_stock} Price Segmented by Common Market Regimes")
axes[0].set_ylabel("Price (INR)")
axes[0].legend(loc="upper left")
axes[0].grid(True, linestyle="--", alpha=0.5)

# Plot 2: Stock Log Returns
axes[1].plot(plot_df.index, plot_df["Log_Return"], color="purple", alpha=0.7, label="Log Returns")
axes[1].set_title(f"{test_stock} Daily Log Returns")
axes[1].set_ylabel("Returns")
axes[1].legend(loc="upper left")
axes[1].grid(True, linestyle="--", alpha=0.5)

# Plot 3: Rolling Volatility
axes[2].plot(plot_df.index, plot_df["Volatility"], color="blue", label="20-Day Rolling Volatility")
axes[2].set_title(f"{test_stock} Historical Volatility (Backward-Looking)")
axes[2].set_ylabel("Volatility (Std Dev)")
axes[2].legend(loc="upper left")
axes[2].grid(True, linestyle="--", alpha=0.5)

# Plot 4: Momentum
axes[3].plot(plot_df.index, plot_df["Momentum"], color="brown", label="20-Day Momentum")
axes[3].set_title(f"{test_stock} Historical Momentum")
axes[3].set_ylabel("Cumulative Log Return")
axes[3].set_xlabel("Date")
axes[3].legend(loc="upper left")
axes[3].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 2b. Phase 2 - Per-regime parameters, shrinkage, cost function, brute force, simulated annealing (your code)

In [ ]:
import numpy as np, pandas as pd, itertools

# Prepare returns_df from features_df to have clean ticker names as columns
log_return_cols = [f"{stock}_log_return" for stock in SELECTED_UNIVERSE]
returns_df = features_df[log_return_cols].copy()
returns_df.columns = SELECTED_UNIVERSE

# Join the 3 market regime probabilities from market_results
probability_cols = [
    "Prob_Market_Regime_0_Low_Vol",
    "Prob_Market_Regime_1_Med_Vol",
    "Prob_Market_Regime_2_High_Vol"
]
returns_df = returns_df.join(market_results[probability_cols], how="inner")

REGIME_COLS = {0: "Prob_Market_Regime_0_Low_Vol",
               1: "Prob_Market_Regime_1_Med_Vol",
               2: "Prob_Market_Regime_2_High_Vol"}
LAMBDA_R = {0: 1.0, 1: 3.0, 2: 6.0}   # risk aversion per regime (your design choice, tune on train only)
K = 5
KAPPA = 0.002                          # cost per flipped holding flag (placeholder, tune)
TRADING_DAYS = 252
stocks = list(SELECTED_UNIVERSE)
N = len(stocks)

def weighted_stats(R, w):
    w = w / w.sum()
    mu = w @ R
    X = R - mu
    S = (X * w[:, None]).T @ X
    return mu, S, X, w

def lw_shrink_identity(S, X, w):
    """Ledoit-Wolf shrinkage toward (avg variance * I), generalized to weighted rows."""
    p = S.shape[0]
    F = (np.trace(S) / p) * np.eye(p)
    d2 = np.sum((S - F) ** 2) / p
    sq = np.einsum('ti,ti->t', X, X)
    xSx = np.einsum('ti,ij,tj->t', X, S, X)
    dev = sq**2 - 2 * xSx + np.sum(S**2)          # ||x x^T - S||_F^2 per day
    b2 = min(np.sum(w**2 * dev) / p, d2)
    delta = b2 / d2 if d2 > 0 else 1.0
    return (1 - delta) * S + delta * F, delta

def fit_regime_params(returns_df, asof):
    """Uses data up to and including `asof` only."""
    hist = returns_df.loc[:asof]
    R = np.expm1(hist[stocks].to_numpy())          # log -> simple returns (portfolio returns add in simple terms)
    out = {}
    for r, col in REGIME_COLS.items():
        mu, S, X, wn = weighted_stats(R, hist[col].to_numpy())
        S_shr, delta = lw_shrink_identity(S, X, wn)
        out[r] = dict(mu=mu * TRADING_DAYS, Sigma=S_shr * TRADING_DAYS,
                      delta=delta, n_eff=1.0 / np.sum(wn**2))
    return out

def blend(fit, p):
    p = np.asarray(p, float); p = p / p.sum()
    mu_t = sum(p[r] * fit[r]["mu"] for r in REGIME_COLS)
    Sigma_t = sum(p[r] * fit[r]["Sigma"] for r in REGIME_COLS)
    lam_t = sum(p[r] * LAMBDA_R[r] for r in REGIME_COLS)
    return mu_t, Sigma_t, lam_t

def build_objective(mu_t, Sigma_t, lam_t, K, kappa, x_prev):
    # cost(x) = lam * w'Σw - mu'w + kappa * sum (x - x_prev)^2, with w = x / K
    Q = lam_t / K**2 * Sigma_t
    lin = -mu_t / K + kappa * (1 - 2 * x_prev)    # turnover term folded in (binary: x^2 = x)
    const = kappa * np.sum(x_prev)
    def cost(X):
        X = np.atleast_2d(X).astype(float)
        c = np.einsum('mi,ij,mj->m', X, Q, X) + X @ lin + const
        return c if c.size > 1 else c[0]
    return dict(Q=Q, lin=lin, const=const, cost=cost, K=K)

def brute_force(obj, N, K):
    combos = np.array(list(itertools.combinations(range(N), K)))
    X = np.zeros((len(combos), N))
    X[np.arange(len(combos))[:, None], combos] = 1
    costs = obj["cost"](X)
    i = costs.argmin()
    return dict(x=X[i], cost=costs[i], all_costs=costs)

def approx_ratio(c_found, all_costs):
    return (all_costs.max() - c_found) / (all_costs.max() - all_costs.min())

def _swap(x, rng):
    y = x.copy()
    i = rng.choice(np.flatnonzero(y == 1)); j = rng.choice(np.flatnonzero(y == 0))
    y[i], y[j] = 0, 1
    return y

def simulated_annealing(obj, N, K, n_iter=5000, seed=0, t_end_frac=1e-3):
    rng = np.random.default_rng(seed); cost = obj["cost"]
    x = np.zeros(N); x[rng.choice(N, K, replace=False)] = 1
    c = cost(x)
    T = max(np.mean([abs(cost(_swap(x, rng)) - c) for _ in range(200)]), 1e-9)
    alpha = t_end_frac ** (1.0 / n_iter)
    best_x, best_c = x.copy(), c
    for _ in range(n_iter):
        y = _swap(x, rng); cy = cost(y); d = cy - c
        if d < 0 or rng.random() < np.exp(-d / T):
            x, c = y, cy
            if c < best_c: best_x, best_c = x.copy(), c
        T *= alpha
    return dict(x=best_x, cost=best_c)

def solve_at(asof, returns_df, x_prev=None, solver="brute", **kw):
    fit = fit_regime_params(returns_df, asof)
    p = returns_df.loc[asof, list(REGIME_COLS.values())].to_numpy(float)
    mu_t, Sigma_t, lam_t = blend(fit, p)
    x_prev = np.zeros(N) if x_prev is None else x_prev
    obj = build_objective(mu_t, Sigma_t, lam_t, K, KAPPA, x_prev)
    res = brute_force(obj, N, K) if solver == "brute" else simulated_annealing(obj, N, K, **kw)
    res["picked"] = [s for s, v in zip(stocks, res["x"]) if v == 1]
    res["obj"], res["fit"], res["p"] = obj, fit, p
    return res

asof = returns_df.index[-1]
bf = solve_at(asof, returns_df)
print("Optimized selection at asof:", bf["picked"], bf["cost"])

# 1. probabilities sum to 1
assert np.allclose(returns_df[list(REGIME_COLS.values())].sum(axis=1), 1, atol=1e-6)

# 2. covariance sanity, plus regime ordering by volatility
fit = bf["fit"]
for r in REGIME_COLS:
    S_val = fit[r]["Sigma"]
    print(r, "sym:", np.allclose(S_val, S_val.T), "min eig:", np.linalg.eigvalsh(S_val).min().round(6),
          "delta:", round(fit[r]["delta"], 3), "n_eff:", round(fit[r]["n_eff"]),
          "avg vol:", np.sqrt(np.diag(S_val)).mean().round(3))

# 3. QUBO expansion matches the explicit formula on every subset
obj = bf["obj"]; x_prev = np.zeros(N); x_prev[:K] = 1
mu_t, Sigma_t, lam_t = blend(fit, bf["p"])
o2 = build_objective(mu_t, Sigma_t, lam_t, K, KAPPA, x_prev)
for c in itertools.combinations(range(N), K):
    x = np.zeros(N); x[list(c)] = 1; w = x / K
    direct = lam_t * w @ Sigma_t @ w - mu_t @ w + KAPPA * np.sum((x - x_prev) ** 2)
    assert np.isclose(o2["cost"](x), direct)

# 4. SA never beats brute force, and how often it finds the optimum
bf2 = brute_force(o2, N, K)
sa = [simulated_annealing(o2, N, K, seed=s) for s in range(20)]
assert all(r["cost"] >= bf2["cost"] - 1e-12 for r in sa)
print("SA hit rate:", np.mean([np.isclose(r["cost"], bf2["cost"]) for r in sa]),
      "mean AR:", np.mean([approx_ratio(r["cost"], bf2["all_costs"]) for r in sa]))

### Ising conversion with the K-constraint penalty (your code)

In [ ]:
# Store the cost function in another variable
obj_dict = bf["obj"]

def build_ising_hamiltonian(obj_dict, P=50.0):
    Q = obj_dict['Q']
    lin = obj_dict['lin']
    const = obj_dict['const']
    K = obj_dict['K']
    N = len(lin)

    # 1. Fold penalty P * (sum(x) - K)^2 into QUBO terms
    Q_tot = Q + P * np.ones((N, N))
    lin_tot = lin - 2.0 * P * K * np.ones(N)
    const_tot = const + P * (K ** 2)

    # Symmetrize total quadratic matrix
    Q_sym = (Q_tot + Q_tot.T) / 2.0

    # 2. Convert x_i -> (1 - z_i) / 2
    h = np.zeros(N)
    J = np.zeros((N, N))
    C_offset = float(const_tot)

    # Process linear and diagonal terms (since x_i^2 = x_i)
    for i in range(N):
        l_eff = lin_tot[i] + Q_sym[i, i]
        h[i] -= 0.5 * l_eff
        C_offset += 0.5 * l_eff

    # Process off-diagonal quadratic pair terms
    for i in range(N):
        for j in range(i + 1, N):
            q_val = 2.0 * Q_sym[i, j]  # Pair interaction weight

            J[i, j] = q_val / 4.0
            J[j, i] = J[i, j]

            h[i] -= q_val / 4.0
            h[j] -= q_val / 4.0
            C_offset += q_val / 4.0

    return h, J, C_offset

build_ising_hamiltonian(obj_dict)

## Phase 3A - Leak-free walk-forward regime inference

In [ ]:
# ======================= PHASE 3A: LEAK-FREE WALK-FORWARD REGIMES =======================
import numpy as np, pandas as pd, itertools, time
from scipy.stats import norm
from hmmlearn import hmm

PROB_COLS = [REGIME_COLS[r] for r in range(3)]
MIN_TRAIN, REFIT_EVERY, REBAL_EVERY = 504, 63, 21     # ~2y initial train, quarterly refit, monthly rebalance
COST_BPS = 10.0                                        # one-way cost per unit of traded weight (10 bps)

mkt_ret = returns_df[stocks].mean(axis=1)              # equal-weight market proxy (same series your HMM used)

def fit_market_hmm(x_train, n_init=3, seed=42):
    best = None
    for s in range(n_init):
        m = hmm.GaussianHMM(3, covariance_type="diag", n_iter=200, random_state=seed + s).fit(x_train)
        ll = m.score(x_train)
        if best is None or ll > best[0]: best = (ll, m)
    m = best[1]
    var = np.asarray(m.covars_).reshape(3, -1)[:, 0]
    return m, np.argsort(var)                          # order states low -> high variance (a model parameter, not data)

def filtered_posteriors(m, x, order):
    """Forward algorithm: p_t = P(s_t | x_1..x_t) with frozen parameters. No future data (unlike predict_proba)."""
    var = np.asarray(m.covars_).reshape(3, -1)[:, 0][order]
    mean = m.means_.reshape(3, -1)[:, 0][order]
    A = m.transmat_[np.ix_(order, order)]; pi = m.startprob_[order]
    lik = norm.pdf(np.asarray(x).reshape(-1, 1), mean, np.sqrt(var))     # (T,3)
    out = np.zeros_like(lik); a = pi * lik[0]; out[0] = a / a.sum()
    for t in range(1, len(lik)):
        a = (out[t - 1] @ A) * lik[t]; s = a.sum()
        out[t] = a / s if s > 0 else out[t - 1] @ A
    return out

def hmm_panel(train_end, block_end):
    """Probabilities for rows [0, block_end) from an HMM trained on rows [0, train_end) only."""
    x = mkt_ret.to_numpy()[:, None]
    m, order = fit_market_hmm(x[:train_end])
    P = filtered_posteriors(m, x[:block_end], order)
    return pd.DataFrame(P, index=returns_df.index[:block_end], columns=PROB_COLS)

def threshold_panel(train_end, block_end, win=20, q=(1/3, 2/3), min_periods=60):
    """Non-quantum ablation: expanding-quantile volatility buckets (past-only), one-hot probabilities."""
    vol = mkt_ret.rolling(win, min_periods=5).std()
    lo = vol.expanding(min_periods).quantile(q[0]); hi = vol.expanding(min_periods).quantile(q[1])
    reg = pd.Series(1, index=vol.index); reg[vol <= lo] = 0; reg[vol >= hi] = 2
    P = np.eye(3)[reg.to_numpy()]
    P[:min_periods] = 1 / 3
    return pd.DataFrame(P[:block_end], index=returns_df.index[:block_end], columns=PROB_COLS)

def refit_points(n, min_train=MIN_TRAIN, every=REFIT_EVERY):
    pts = list(range(min_train, n, every)); return pts

def build_panels(panel_fn, n=None):
    """Dict: refit_point R -> prob panel over rows [0, next_refit) from a model trained on rows [0, R)."""
    n = n or len(returns_df); pts = refit_points(n); panels = {}
    for k, R in enumerate(pts):
        end = pts[k + 1] if k + 1 < len(pts) else n
        panels[R] = panel_fn(R, end)
    return panels


## Phase 3B - Solver-agnostic backtest harness

In [ ]:
# ======================= PHASE 3B: SOLVER-AGNOSTIC BACKTEST HARNESS =======================
def make_objective_at(df_probs, i, x_prev):
    """Objective for the rebalance at row i using ONLY rows <= i (reuses your fit_regime_params / blend / build_objective)."""
    asof = df_probs.index[i]
    fit = fit_regime_params(df_probs, asof)
    p = df_probs.iloc[i][PROB_COLS].to_numpy(float)
    mu_t, Sigma_t, lam_t = blend(fit, p)
    return build_objective(mu_t, Sigma_t, lam_t, K, KAPPA, x_prev), dict(fit=fit, p=p, mu=mu_t, Sigma=Sigma_t, lam=lam_t)

def solver_brute(obj, ctx): return brute_force(obj, N, K)["x"]
def solver_sa(obj, ctx, n_iter=3000): return simulated_annealing(obj, N, K, n_iter=n_iter, seed=ctx["i"])["x"]

def run_backtest(panels, solver, label, rebal_every=REBAL_EVERY, cost_bps=COST_BPS, verbose=False):
    n = len(returns_df); pts = sorted(panels); R_simple = np.expm1(returns_df[stocks].to_numpy())
    dates = list(range(pts[0], n - 1, rebal_every))
    x_prev = np.zeros(N); w_prev = np.zeros(N)
    daily = pd.Series(0.0, index=returns_df.index); rec = []
    for i in dates:
        Rb = max(p for p in pts if p <= i)                          # latest refit available at date i
        df_p = returns_df[stocks].join(panels[Rb], how="inner")
        obj, ctx = make_objective_at(df_p, i, x_prev); ctx["i"] = i; ctx["refit"] = Rb
        x = np.asarray(solver(obj, ctx), float)
        assert x.sum() == K, "solver returned an infeasible portfolio"
        w = x / K; turnover = np.abs(w - w_prev).sum()
        j_end = min(i + rebal_every, n - 1)
        r = R_simple[i + 1: j_end + 1] @ w                           # hold from i+1 to next rebalance
        r = r.copy(); r[0] -= cost_bps / 1e4 * turnover               # cost paid on the first holding day
        daily.iloc[i + 1: j_end + 1] = r
        rec.append(dict(date=returns_df.index[i], turnover=turnover, picked=tuple(np.flatnonzero(x)), n_changed=int(np.abs(x - x_prev).sum() / 2)))
        x_prev, w_prev = x, w
    daily = daily.iloc[dates[0] + 1:]
    return dict(label=label, daily=daily, rebal=pd.DataFrame(rec), panels_used=pts)

def benchmark_equal_weight(start_row, rebal_every=REBAL_EVERY, cost_bps=COST_BPS):
    n = len(returns_df); R_simple = np.expm1(returns_df[stocks].to_numpy()); w = np.ones(N) / N
    daily = pd.Series(R_simple[start_row + 1:] @ w, index=returns_df.index[start_row + 1:])
    return dict(label="Equal-weight 1/N (all 11)", daily=daily, rebal=pd.DataFrame(dict(turnover=[0.0])))

def metrics(res, bench_daily):
    r = res["daily"]; ann = 252
    sharpe = r.mean() / r.std() * np.sqrt(ann) if r.std() > 0 else np.nan
    eq = (1 + r).cumprod(); mdd = (eq / eq.cummax() - 1).min()
    te = (r - bench_daily.reindex(r.index)).std() * np.sqrt(ann)
    return dict(Strategy=res["label"], Sharpe=round(sharpe, 3), AnnReturn=round((1 + r).prod() ** (ann / len(r)) - 1, 4),
                MaxDD=round(mdd, 3), TrackErr=round(te, 4), AvgTurnover=round(res["rebal"]["turnover"].mean(), 3))

def summarize(results, bench):
    return pd.DataFrame([metrics(r, bench["daily"]) for r in results + [bench]]).set_index("Strategy")


### Run Phase 3: panels, leakage tests, classical backtests

In [ ]:
import matplotlib.pyplot as plt
t0 = time.time()
panels_hmm = build_panels(hmm_panel)          # refit every 63 days on rows < R, filtered posteriors only
panels_thr = build_panels(threshold_panel)
print(f"{len(panels_hmm)} refit points, built in {time.time()-t0:.1f}s")

# ---- leakage tests ----
def leakage_tests():
    global mkt_ret
    R = sorted(panels_hmm)[2]
    base = hmm_panel(R, R + 63).iloc[:R + 5]
    orig = mkt_ret.copy(); mkt_ret = orig.copy(); mkt_ret.iloc[R + 5:] += 0.05            # perturb the FUTURE (after row R+5)
    alt = hmm_panel(R, R + 63).iloc[:R + 5]; mkt_ret = orig
    d_filtered = float(np.abs(alt.values - base.values).max())
    x = orig.to_numpy()[:, None]; x2 = x.copy(); x2[R + 5:] += 0.05
    m1 = hmm.GaussianHMM(3, covariance_type="diag", n_iter=100, random_state=42).fit(x[:R])
    d_smoothed = float(np.abs(m1.predict_proba(x[:R + 63])[:R + 5] - m1.predict_proba(x2[:R + 63])[:R + 5]).max())
    print(f"Filtered posteriors: max change in past rows when the future changes = {d_filtered:.2e}  (must be 0)")
    print(f"Smoothed (predict_proba): max change in past rows when the future changes = {d_smoothed:.3f}  (leakage)")
    assert d_filtered < 1e-12 and d_smoothed > 1e-6
leakage_tests()

# ---- classical backtests (brute force = exact optimum, SA = heuristic) ----
bt_bf = run_backtest(panels_hmm, solver_brute, "HMM + Brute force")
bt_sa = run_backtest(panels_hmm, solver_sa, "HMM + Simulated annealing")
bench = benchmark_equal_weight(sorted(panels_hmm)[0])
tab3 = summarize([bt_bf, bt_sa], bench); display(tab3)

fig, ax = plt.subplots(figsize=(11, 4))
for r in (bt_bf, bt_sa, bench): ax.plot((1 + r["daily"]).cumprod(), label=r["label"])
ax.set_title("Out-of-sample equity curves (net of costs)"); ax.legend(); ax.grid(alpha=.3); plt.show()

## Phase 4 - Ising verification, fast simulator, Qiskit circuits, penalty QAOA baseline
Conventions: qubit *i* = asset *i* = bit *i* of the bitstring index (Qiskit little-endian), `x = (1 - z)/2`, cost layer `exp(-i*gamma*H_C)`, mixer `exp(-i*beta*sum X)` or the XY ring. Optimisation runs on a fast exact numpy simulator that is **cross-checked against Qiskit `Statevector`** below.

In [ ]:
# ======================= PHASE 4: ISING VERIFICATION, FAST SIMULATOR, QISKIT CIRCUITS, PENALTY QAOA =======================
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import XXPlusYYGate, RYGate
from qiskit.quantum_info import Statevector
from scipy.optimize import minimize

BITS = None
def _bits(n):
    global BITS
    if BITS is None or BITS.shape[1] != n:
        b = np.arange(2 ** n)
        BITS = ((b[:, None] >> np.arange(n)) & 1).astype(np.int8)       # BITS[b, i] = x_i (qubit i = bit i, Qiskit little-endian)
    return BITS

def ising_energies(h, J, c):
    """Energy of every bitstring: E = sum h_i z_i + sum_{i<j} J_ij z_i z_j + c, with z = 1 - 2x."""
    Z = 1.0 - 2.0 * _bits(len(h))
    return Z @ h + 0.5 * np.einsum('bi,ij,bj->b', Z, J, Z) + c

def normalize_obj(obj, n_samples=2000, seed=0):
    """z-score the cost over RANDOM FEASIBLE subsets (no brute force needed) so gamma ranges are comparable across dates."""
    rng = np.random.default_rng(seed)
    X = np.zeros((n_samples, N))
    for s in range(n_samples): X[s, rng.choice(N, K, replace=False)] = 1
    c = obj["cost"](X); mu, sd = c.mean(), c.std()
    return dict(Q=obj["Q"] / sd, lin=obj["lin"] / sd, const=(obj["const"] - mu) / sd, K=obj["K"]), (mu, sd)

class Instance:
    """One portfolio-selection instance in QAOA form. Energies are z-scored: a random feasible subset has mean 0, std 1."""
    def __init__(self, obj, P_pen=4.0):
        self.obj = obj; self.nobj, self.norm = normalize_obj(obj)
        self.h, self.J, self.c = build_ising_hamiltonian(self.nobj, P=0.0)           # constraint handled by XY mixer
        self.hp, self.Jp, self.cp = build_ising_hamiltonian(self.nobj, P=P_pen)      # penalty version for the baseline
        self.E_xy = ising_energies(self.h, self.J, self.c); self.E_pen = ising_energies(self.hp, self.Jp, self.cp)
        self.feas = _bits(N).sum(1) == K
        Ef = np.where(self.feas, self.E_xy, np.inf)
        self.opt = int(np.argmin(Ef)); self.Emin = Ef[self.opt]
        self.Emax = self.E_xy[self.feas].max()
        self.P_pen = P_pen
    def AR(self, E):  return (self.Emax - E) / (self.Emax - self.Emin)               # 1 = optimal, 0 = worst feasible
    def bitstring(self, b): return _bits(N)[b].astype(float)

def verify_ising(inst, n_check=300, seed=1):
    rng = np.random.default_rng(seed); B = _bits(N)
    for b in rng.integers(0, 2 ** N, n_check):
        x = B[b].astype(float)
        assert np.isclose(inst.E_xy[b], x @ inst.nobj["Q"] @ x + x @ inst.nobj["lin"] + inst.nobj["const"]), "Ising != QUBO"
        pen = inst.P_pen * (x.sum() - K) ** 2
        assert np.isclose(inst.E_pen[b], inst.E_xy[b] + pen), "penalty Ising != QUBO + penalty"
    assert np.isclose(inst.Emin, (brute_force(inst.obj, N, K)["cost"] - inst.norm[0]) / inst.norm[1]), "optimum mismatch vs brute force"
    return True

# ---------------- fast exact simulator (numpy), cross-checked against Qiskit below ----------------
def ring_edges(n):
    e = [(i, i + 1) for i in range(0, n - 1, 2)] + [(i, i + 1) for i in range(1, n - 1, 2)]
    return e + [(n - 1, 0)] if n > 2 else e
_PAIR_IDX = {}
def _pair_idx(n, i, j):
    key = (n, i, j)
    if key not in _PAIR_IDX:
        b = np.arange(2 ** n); bi = (b >> i) & 1; bj = (b >> j) & 1
        _PAIR_IDX[key] = (b[(bi == 1) & (bj == 0)], b[(bi == 0) & (bj == 1)])
    return _PAIR_IDX[key]

def apply_xy(psi, i, j, beta):
    """exp(-i beta (XX+YY)/2) on qubits i,j: rotates within span{|01>,|10>}, leaves |00>,|11>. Preserves Hamming weight."""
    a_idx, b_idx = _pair_idx(N, i, j); a, b = psi[a_idx], psi[b_idx]
    c, s = np.cos(beta), -1j * np.sin(beta)
    psi[a_idx] = c * a + s * b; psi[b_idx] = c * b + s * a

def apply_rx_all(psi, beta):
    c, s = np.cos(beta), -1j * np.sin(beta); t = psi.reshape((2,) * N)
    for ax in range(N):
        a = np.take(t, 0, axis=ax); b = np.take(t, 1, axis=ax)
        t = np.stack([c * a + s * b, s * a + c * b], axis=ax)
    return t.reshape(-1)

DICKE = None
def dicke_state():
    v = (_bits(N).sum(1) == K).astype(complex); return v / np.linalg.norm(v)

def qaoa_probs(theta, p, inst, mode):
    """theta = [gamma_1..gamma_p, beta_1..beta_p]. mode 'xy': Dicke start + XY ring mixer on E_xy.  mode 'penalty': |+>^N + RX mixer on E_pen."""
    g, b = theta[:p], theta[p:]
    if mode == "xy": psi = dicke_state(); E = inst.E_xy
    else: psi = np.full(2 ** N, 2 ** (-N / 2), dtype=complex); E = inst.E_pen
    edges = ring_edges(N)
    for l in range(p):
        psi = psi * np.exp(-1j * g[l] * E)
        if mode == "xy":
            psi = psi.copy()
            for (i, j) in edges: apply_xy(psi, i, j, b[l])
        else: psi = apply_rx_all(psi, b[l])
    pr = np.abs(psi) ** 2; return pr / pr.sum()

def cvar(E, pr, alpha):
    """Mean energy of the best alpha-fraction of probability mass. alpha=1 -> ordinary expectation."""
    if alpha >= 1: return float(pr @ E)
    o = np.argsort(E, kind="stable"); cum = np.cumsum(pr[o]); take = np.minimum(pr[o], np.clip(alpha - (cum - pr[o]), 0, None))
    return float(take @ E[o] / alpha)

# ---------------- Qiskit circuits (for resource counts, hardware, and cross-checking the fast simulator) ----------------
def dicke_circuit(n, k):
    """Bartschi-Eidenbenz deterministic Dicke-state preparation. Qubits are 1-indexed in the paper; qubit q <-> Qiskit index q-1."""
    qc = QuantumCircuit(n)
    for q in range(n - k + 1, n + 1): qc.x(q - 1)                       # |0^{n-k} 1^k>
    def scs(m, kk):                                                       # SCS_{m,kk} acts on qubits m-kk..m
        # gate (i): qubits m-1, m
        qc.cx(m - 2, m - 1); qc.append(RYGate(2 * np.arccos(np.sqrt(1.0 / m))).control(1), [m - 1, m - 2]); qc.cx(m - 2, m - 1)
        for l in range(2, kk + 1):                                        # gates (ii)_l
            qc.cx(m - l - 1, m - 1)
            qc.append(RYGate(2 * np.arccos(np.sqrt(l / m))).control(2), [m - 1, m - l, m - l - 1])
            qc.cx(m - l - 1, m - 1)
    for m in range(n, k, -1): scs(m, k)
    for m in range(k, 1, -1): scs(m, m - 1)
    return qc

def qaoa_circuit(theta, p, inst, mode):
    g, b = theta[:p], theta[p:]; qc = QuantumCircuit(N)
    h, J, E = (inst.h, inst.J, "xy") if mode == "xy" else (inst.hp, inst.Jp, "pen")
    if mode == "xy": qc.compose(dicke_circuit(N, K), inplace=True)
    else: qc.h(range(N))
    for l in range(p):
        for i in range(N):
            if abs(h[i]) > 1e-12: qc.rz(2 * g[l] * h[i], i)
        for i in range(N):
            for j in range(i + 1, N):
                if abs(J[i, j]) > 1e-12: qc.rzz(2 * g[l] * J[i, j], i, j)
        if mode == "xy":
            for (i, j) in ring_edges(N): qc.append(XXPlusYYGate(2 * b[l]), [i, j])
        else: qc.rx(2 * b[l], range(N))
    return qc

def resources(theta, p, inst, mode):
    qc = qaoa_circuit(theta, p, inst, mode)
    tq = transpile(qc, basis_gates=["cx", "rz", "sx", "x"], optimization_level=1, seed_transpiler=0)
    return dict(depth=tq.depth(), cx=tq.count_ops().get("cx", 0), total_gates=sum(tq.count_ops().values()))

def crosscheck_qiskit(inst, p=2, seed=0):
    rng = np.random.default_rng(seed); th = rng.uniform(0.1, 1.0, 2 * p)
    D = Statevector.from_instruction(dicke_circuit(N, K)).data
    assert np.allclose(np.abs(D) ** 2, np.abs(dicke_state()) ** 2, atol=1e-9), "Dicke circuit != Dicke state"
    for mode in ("xy", "penalty"):
        pq = np.abs(Statevector.from_instruction(qaoa_circuit(th, p, inst, mode)).data) ** 2
        assert np.allclose(pq, qaoa_probs(th, p, inst, mode), atol=1e-8), f"Qiskit circuit != numpy simulator ({mode})"
    return True


In [ ]:
# pick one real instance (a rebalance date late in the sample) and verify everything
Rb = sorted(panels_hmm)[3]; i0 = 700
df_p = returns_df[stocks].join(panels_hmm[Rb], how="inner")
obj0, ctx0 = make_objective_at(df_p, i0, np.zeros(N))
inst0 = Instance(obj0)
print("Ising energies == QUBO cost (and penalty version == cost + P*(sum x - K)^2), optimum == brute force:", verify_ising(inst0))
print("Qiskit circuits == numpy simulator (Dicke state, XY-QAOA, penalty-QAOA):", crosscheck_qiskit(inst0))
d = dicke_circuit(N, K); tq = transpile(d, basis_gates=["cx", "rz", "sx", "x"], optimization_level=1, seed_transpiler=0)
print(f"Dicke-state prep alone: depth {tq.depth()}, CX count {tq.count_ops().get('cx', 0)}")

## Phase 5 - Constraint-preserving CVaR-QAOA and the RQ2 experiment
Four configurations are compared: {penalty + X mixer, Dicke + XY mixer} x {expectation, CVaR}. **A uniform-random-feasible-sampling baseline is included on purpose**: at N = 11 there are only 462 feasible subsets, so "best of many shots" is easy for any sampler. The informative numbers are `p_opt` (lift over 1/462), `exp_AR`, and `feasible_prob`.

In [ ]:
# ======================= PHASE 5: CVaR-QAOA OPTIMIZER, METRICS, RQ2 EXPERIMENT =======================
ALPHA = 0.2          # CVaR fraction (alpha=1 -> plain expectation)
GAMMA_MAX, BETA_MAX = 2.0, 1.0

def run_qaoa(inst, mode="xy", p=2, alpha=ALPHA, restarts=3, seed=0, init=None, maxiter=200, shots=None):
    """COBYLA over (gammas, betas). `init` = list of starting points (warm start); otherwise random restarts.
    shots=None -> exact probabilities inside the optimizer; shots=int -> finite-shot CVaR (noisy objective)."""
    E = inst.E_xy if mode == "xy" else inst.E_pen
    rng = np.random.default_rng(seed); best = None; nfev_total = 0
    starts = [np.asarray(s, float) for s in init] if init is not None else \
             [np.r_[rng.uniform(0, GAMMA_MAX, p), rng.uniform(0, BETA_MAX, p)] for _ in range(restarts)]
    for th0 in starts:
        hist = []
        def f(th):
            pr = qaoa_probs(th, p, inst, mode)
            if shots: pr = rng.multinomial(shots, pr) / shots
            hist.append(float(pr @ E)); return cvar(E, pr, alpha)
        r = minimize(f, th0, method="COBYLA", options=dict(maxiter=maxiter, rhobeg=0.3, tol=1e-4))
        nfev_total += r.nfev
        if best is None or r.fun < best["fun"]: best = dict(fun=r.fun, theta=r.x, hist=hist, nfev=r.nfev)
    best["nfev_total"] = nfev_total; best["mode"] = mode; best["p"] = p
    return best

def evaluate(inst, theta, p, mode, shots=64, seed=0):
    """Quality metrics from the final state."""
    pr = qaoa_probs(theta, p, inst, mode); feas = inst.feas
    pf = pr[feas].sum(); AR_b = inst.AR(inst.E_xy)
    exp_AR = float((pr[feas] * AR_b[feas]).sum() / pf) if pf > 0 else 0.0
    rng = np.random.default_rng(seed); s = rng.choice(2 ** N, size=shots, p=pr); sf = s[feas[s]]
    best_AR = float(AR_b[sf].max()) if len(sf) else 0.0
    best_b = int(sf[np.argmin(inst.E_xy[sf])]) if len(sf) else None
    return dict(feasible_prob=float(pf), p_opt=float(pr[inst.opt]), exp_AR=exp_AR, best_of_shots_AR=best_AR, best_b=best_b)

def random_baseline(inst, shots=64, trials=300, seed=0):
    """Uniform sampling over feasible subsets: the bar any quantum sampler has to clear."""
    rng = np.random.default_rng(seed); AR_b = inst.AR(inst.E_xy); idx = np.flatnonzero(inst.feas)
    best = [AR_b[rng.choice(idx, shots)].max() for _ in range(trials)]
    return dict(feasible_prob=1.0, p_opt=1 / len(idx), exp_AR=float(AR_b[idx].mean()), best_of_shots_AR=float(np.mean(best)))

def select_from_samples(inst, theta, p, mode, shots=1024, seed=0):
    """What you would do on hardware: sample, keep feasible bitstrings, return the lowest-cost one."""
    pr = qaoa_probs(theta, p, inst, mode); s = np.random.default_rng(seed).choice(2 ** N, size=shots, p=pr)
    sf = s[inst.feas[s]]
    return inst.bitstring(int(sf[np.argmin(inst.E_xy[sf])])) if len(sf) else None

def gap_curve(inst, hist):
    """Normalized gap of expected energy: 1 = random feasible subset (E=0), 0 = optimum."""
    return (np.array(hist) - inst.Emin) / (0 - inst.Emin)

def evals_to_gap(inst, hist, target=0.5):
    g = gap_curve(inst, hist); k = np.flatnonzero(g <= target)
    return int(k[0] + 1) if len(k) else np.nan

def sample_instances(n_inst=6, seed=0):
    """Real instances from the backtest (spread over the test period), built with the leak-free regime panels."""
    pts = sorted(panels_hmm); n = len(returns_df); dates = list(range(pts[0], n - 1, REBAL_EVERY))
    pick = [dates[int(i)] for i in np.linspace(0, len(dates) - 1, n_inst)]
    out = []
    for i in pick:
        Rb = max(q for q in pts if q <= i); df_p = returns_df[stocks].join(panels_hmm[Rb], how="inner")
        rng = np.random.default_rng(i); xp = np.zeros(N); xp[rng.choice(N, K, replace=False)] = 1
        obj, ctx = make_objective_at(df_p, i, xp); out.append((i, Instance(obj), ctx))
    return out

CONFIGS = {"Penalty + X mixer, expectation": ("penalty", 1.0), "Penalty + X mixer, CVaR": ("penalty", ALPHA),
           "Dicke + XY mixer, expectation": ("xy", 1.0), "Dicke + XY mixer, CVaR": ("xy", ALPHA)}

def rq2_experiment(n_inst=5, n_seeds=2, depths=(1, 2, 3), restarts=2):
    rows = []; insts = sample_instances(n_inst)
    for (i, inst, ctx) in insts:
        for name, (mode, a) in CONFIGS.items():
            for p in depths:
                for s in range(n_seeds):
                    r = run_qaoa(inst, mode, p, a, restarts=restarts, seed=1000 * s + i)
                    m = evaluate(inst, r["theta"], p, mode, seed=s)
                    rows.append(dict(date_row=i, config=name, p=p, seed=s, nfev=r["nfev_total"], **m))
        rb = random_baseline(inst)
        for p in depths: rows.append(dict(date_row=i, config="Random feasible sampling (baseline)", p=p, seed=0, nfev=0, **rb))
    return pd.DataFrame(rows)

def rq2_summary(df):
    g = df.groupby(["config", "p"])[["feasible_prob", "p_opt", "exp_AR", "best_of_shots_AR", "nfev"]].mean().round(3)
    g["opt_lift_vs_random"] = (g["p_opt"] / (1 / 462)).round(1)      # 462 = C(11,5) feasible subsets
    return g


In [ ]:
t0 = time.time()
rq2 = rq2_experiment()           # defaults: 5 real instances x 2 seeds x depths 1-3 x 4 configs (scale up for the final study)
print(f"done in {time.time()-t0:.0f}s")
tab5 = rq2_summary(rq2); display(tab5)

# circuit resources after transpilation (angles don't change gate counts)
th_demo = np.array([.3] * 3 + [.4] * 3); rows = []
for p in (1, 2, 3):
    for mode, name in (("penalty", "Penalty + X mixer"), ("xy", "Dicke + XY mixer")):
        rows.append(dict(config=name, p=p, **resources(th_demo[:p].tolist() + th_demo[3:3 + p].tolist(), p, inst0, mode)))
tab5_res = pd.DataFrame(rows).set_index(["config", "p"]); display(tab5_res)

## Phase 6 - Regime-conditioned warm starts (RQ3)
Per refit point, QAOA angles are optimised on one archetypal instance **per regime** (regime mean/covariance/lambda, built from training rows only). At each rebalance the start point is the library angles of the dominant regime, the posterior-weighted blend, or the previous date's angles - all against cold random starts. `evals_to_gap50` = objective evaluations until the expected energy closes half the gap between a random feasible subset and the optimum.

In [ ]:
# ======================= PHASE 6: REGIME-KEYED WARM STARTS (RQ3) =======================
P_DEPTH = 2

def build_library(panels, p=P_DEPTH, alpha=ALPHA, restarts=4):
    """For each refit point R: optimise QAOA angles on one archetypal instance PER REGIME, using ONLY data before R."""
    lib = {}
    for R, panel in panels.items():
        df_p = returns_df[stocks].join(panel, how="inner")
        fit = fit_regime_params(df_p.iloc[:R], returns_df.index[R - 1])       # train rows only
        lib[R] = {}
        for r in range(3):
            obj = build_objective(fit[r]["mu"], fit[r]["Sigma"], LAMBDA_R[r], K, KAPPA, np.zeros(N))
            lib[R][r] = run_qaoa(Instance(obj), "xy", p, alpha, restarts=restarts, seed=R + r)["theta"]
    return lib

def make_qaoa_solver(init="cold", restarts=1, p=P_DEPTH, alpha=ALPHA, maxiter=150, lib=None):
    """init: 'cold' random start(s) | 'dominant' library angles of the most likely regime | 'blend' posterior-weighted library
    angles | 'prev' angles optimised at the previous rebalance. Returns (solver, diagnostics list)."""
    diag, state = [], dict(prev=None)
    def solver(obj, ctx):
        inst = Instance(obj); pr, R = ctx["p"], ctx["refit"]; starts = None
        if init == "dominant": starts = [lib[R][int(np.argmax(pr))]]
        elif init == "blend": starts = [sum(pr[r] * lib[R][r] for r in range(3))]
        elif init == "prev" and state["prev"] is not None: starts = [state["prev"]]
        r = run_qaoa(inst, "xy", p, alpha, restarts=restarts, seed=ctx["i"], init=starts, maxiter=maxiter)
        state["prev"] = r["theta"]
        m = evaluate(inst, r["theta"], p, "xy", seed=ctx["i"])
        diag.append(dict(row=ctx["i"], nfev=r["nfev_total"], evals_to_gap50=evals_to_gap(inst, r["hist"], 0.5),
                         feasible_prob=m["feasible_prob"], p_opt=m["p_opt"], exp_AR=m["exp_AR"], best64_AR=m["best_of_shots_AR"]))
        x = select_from_samples(inst, r["theta"], p, "xy", seed=ctx["i"])
        return x if x is not None else obj_prev_fallback(ctx)
    return solver, diag

def obj_prev_fallback(ctx):                      # never triggered in XY mode (always feasible); safety net
    x = np.zeros(N); x[:K] = 1; return x

def diag_summary(diags):
    rows = {name: pd.DataFrame(d).drop(columns="row").mean().round(3) for name, d in diags.items()}
    return pd.DataFrame(rows).T


In [ ]:
t0 = time.time(); lib = build_library(panels_hmm); print(f"library built in {time.time()-t0:.0f}s ({len(lib)} refit points x 3 regimes)")
specs = {"QAOA cold (1 start)": dict(init="cold", restarts=1), "QAOA cold (3 starts)": dict(init="cold", restarts=3),
         "QAOA warm: dominant regime": dict(init="dominant", lib=lib), "QAOA warm: regime blend": dict(init="blend", lib=lib),
         "QAOA warm: previous date": dict(init="prev")}
qaoa_res, qaoa_diag = [], {}
for name, kw in specs.items():
    t0 = time.time(); solver, dg = make_qaoa_solver(**kw)
    qaoa_res.append(run_backtest(panels_hmm, solver, name)); qaoa_diag[name] = dg; print(f"{name}: {time.time()-t0:.0f}s")
print("\nOptimiser behaviour (means over all rebalance dates; nfev is capped by maxiter so use evals_to_gap50):")
tab6_diag = diag_summary(qaoa_diag); display(tab6_diag)
print("Financial results (the brute-force row is the exact optimum of the same objective):")
tab6_fin = summarize([bt_bf, bt_sa] + qaoa_res, bench); display(tab6_fin)

## Phase 7 - Quantum regime inference with a data re-uploading VQC (RQ1)
3 qubits, 3 re-uploading layers, features `[return, 20d vol, 20d momentum]` standardised with training statistics, readout = `<Z>` per qubit -> softmax over (Low, Med, High) with a trainable scale and biases, temperature-calibrated on the last 20% of each training window. **Training target (documented decision):** `TARGET="hmm_soft"` distils the HMM's filtered posteriors from the same training window, so the quantum model learns the *same* regime definition from richer inputs; `TARGET="realized_vol"` is the independent alternative. Because of the distillation target the VQC cannot claim to *beat* the HMM on regime identification. The fair comparisons are stability, label-free predictive power for forward volatility, and downstream backtest performance, with everything else held fixed. The HQMM stretch goal is not implemented.

In [ ]:
# ======================= PHASE 7: QUANTUM REGIME INFERENCE - DATA RE-UPLOADING VQC (RQ1) =======================
from scipy.stats import spearmanr
NQ, LAYERS, WARMUP = 3, 3, 20
TARGET = "hmm_soft"       # 'hmm_soft': distil the HMM's filtered posteriors (train window only) | 'realized_vol': next-10d vol terciles
FWD = 10

# ---- features available at the close of day t (no look-ahead) ----
_vol = mkt_ret.rolling(20, min_periods=2).std().fillna(0.0); _mom = mkt_ret.rolling(20, min_periods=2).sum().fillna(0.0)
FEATS = np.c_[mkt_ret.to_numpy(), _vol.to_numpy(), _mom.to_numpy()]          # [return, 20d vol, 20d momentum]

# ---- batched numpy statevector simulator for the 3-qubit circuit ----
def _apply1(psi, U, q):
    ax = NQ - q; psi = np.moveaxis(psi, ax, -1)
    out = psi @ U.T if U.ndim == 2 else np.einsum('s...k,sjk->s...j', psi, U)
    return np.moveaxis(out, -1, ax)
def _ry(theta):  c, s = np.cos(theta / 2), np.sin(theta / 2); return np.stack([np.stack([c, -s], -1), np.stack([s, c], -1)], -2)
def _rz(phi):    return np.diag([np.exp(-1j * phi / 2), np.exp(1j * phi / 2)])
def _cz_sign(a, b):
    sh = (1,) + (2,) * NQ; idx = np.indices((2,) * NQ); bit = lambda q: idx[NQ - 1 - q]
    return np.where((bit(a) == 1) & (bit(b) == 1), -1.0, 1.0).reshape(sh)
_CZ = [_cz_sign(0, 1), _cz_sign(1, 2), _cz_sign(2, 0)]

def unpack(th):
    n = LAYERS * NQ; w, b, a, c = [th[i * n:(i + 1) * n].reshape(LAYERS, NQ) for i in range(4)]
    return w, b, a, c, th[4 * n], th[4 * n + 1: 4 * n + 4]

def vqc_expZ(th, X):
    w, b, a, c, s, t = unpack(th); S = len(X)
    psi = np.zeros((S,) + (2,) * NQ, complex); psi[(slice(None),) + (0,) * NQ] = 1.0
    for l in range(LAYERS):
        for q in range(NQ): psi = _apply1(psi, _ry(w[l, q] * X[:, q] + b[l, q]), q)       # data (re-)uploading
        for q in range(NQ): psi = _apply1(psi, _rz(a[l, q]), q); psi = _apply1(psi, _ry(np.full(S, c[l, q])), q)
        for sg in _CZ: psi = psi * sg
    pr = np.abs(psi) ** 2; z = np.zeros((S, NQ))
    for q in range(NQ):
        m = np.moveaxis(pr, NQ - q, -1).reshape(S, -1, 2).sum(1); z[:, q] = m[:, 0] - m[:, 1]
    return z

def vqc_logits(th, X):
    w, b, a, c, s, t = unpack(th); return s * vqc_expZ(th, X) + t
def softmax(L, T=1.0):
    L = L / T; L = L - L.max(1, keepdims=True); e = np.exp(L); return e / e.sum(1, keepdims=True)

def vqc_qiskit_expZ(th, x):
    """Same circuit built in Qiskit - used only to cross-check the fast simulator."""
    from qiskit.quantum_info import SparsePauliOp
    w, b, a, c, s, t = unpack(th); qc = QuantumCircuit(NQ)
    for l in range(LAYERS):
        for q in range(NQ): qc.ry(float(w[l, q] * x[q] + b[l, q]), q)
        for q in range(NQ): qc.rz(float(a[l, q]), q); qc.ry(float(c[l, q]), q)
        qc.cz(0, 1); qc.cz(1, 2); qc.cz(2, 0)
    sv = Statevector.from_instruction(qc)
    return np.array([sv.expectation_value(SparsePauliOp("I" * (NQ - 1 - q) + "Z" + "I" * q)).real for q in range(NQ)])

def train_vqc(X, T, th0, maxiter=40, seed=0):
    ce = lambda th: -np.mean(np.sum(T * np.log(softmax(vqc_logits(th, X)) + 1e-9), axis=1))
    r = minimize(ce, th0, method="L-BFGS-B", options=dict(maxiter=maxiter))
    return r.x, r.fun

def init_params(rng):
    n = LAYERS * NQ
    return np.r_[rng.uniform(.5, 1.5, n), np.zeros(n), rng.uniform(-1, 1, n), rng.uniform(-1, 1, n), 2.0, np.zeros(3)]

def realized_vol_targets(R, q=(1/3, 2/3)):
    """Alternative target: terciles (train-only quantiles) of the NEXT-10-day realised vol. Uses only rows < R."""
    fv = mkt_ret[::-1].rolling(FWD).std()[::-1].shift(-1).to_numpy()          # std of r_{t+1..t+FWD}
    fv = fv[:R]; ok = ~np.isnan(fv); ok[R - FWD:] = False                     # drop rows whose window leaks past R
    lo, hi = np.nanquantile(fv[ok], q); lab = np.where(fv <= lo, 0, np.where(fv >= hi, 2, 1))
    T = np.eye(3)[lab]; return T, ok

_VQC_STATE = dict(th=None)
def vqc_panel(train_end, block_end, restarts=2, max_rows=400):
    R = train_end; rng = np.random.default_rng(R)
    mu = FEATS[WARMUP:R].mean(0); sd = FEATS[WARMUP:R].std(0) + 1e-12
    Z = np.clip((FEATS - mu) / sd, -4, 4)
    if TARGET == "hmm_soft": T = hmm_panel(R, R).to_numpy(); ok = np.arange(R) >= WARMUP     # HMM trained on rows < R
    else: T, ok = realized_vol_targets(R); ok &= np.arange(R) >= WARMUP
    rows = np.flatnonzero(ok); cut = int(len(rows) * 0.8); fit_rows, val_rows = rows[:cut], rows[cut:]
    fit_rows = fit_rows[::max(1, len(fit_rows) // max_rows)]                   # subsample for speed (finite-difference gradients)
    # first refit: a few random starts; later refits: warm start from the previous refit's parameters (much cheaper)
    starts = [_VQC_STATE["th"]] if _VQC_STATE["th"] is not None else [init_params(rng) for _ in range(restarts)]
    best = min((train_vqc(Z[fit_rows], T[fit_rows], th0, maxiter=25 if _VQC_STATE["th"] is not None else 40) for th0 in starts), key=lambda r: r[1]); th = best[0]
    # temperature calibration on the most recent 20% of the training window
    Lv = vqc_logits(th, Z[val_rows]); grid = np.linspace(0.5, 3.0, 26)
    nll = [-np.mean(np.sum(T[val_rows] * np.log(softmax(Lv, g) + 1e-9), 1)) for g in grid]; Tcal = grid[int(np.argmin(nll))]
    _VQC_STATE["th"] = th
    P = softmax(vqc_logits(th, Z[:block_end]), Tcal); P[:WARMUP] = 1 / 3
    return pd.DataFrame(P, index=returns_df.index[:block_end], columns=PROB_COLS)

# ---------------- comparison: regime quality and downstream backtest ----------------
def oos_probs(panels):
    pts = sorted(panels); n = len(returns_df)
    return pd.concat([panels[R].iloc[R:(pts[k + 1] if k + 1 < len(pts) else n)] for k, R in enumerate(pts)])

def regime_report(named_panels, ref="HMM (classical)"):
    fv = mkt_ret[::-1].rolling(FWD).std()[::-1].shift(-1)       # EVALUATION ONLY: realised vol over the next 10 days
    P0 = oos_probs(named_panels[ref]); rows = []
    for name, pan in named_panels.items():
        P = oos_probs(pan); am = P.to_numpy().argmax(1); e = P.to_numpy() @ np.arange(3); f = fv.reindex(P.index)
        ok = ~f.isna(); by = [f[ok][am[ok.to_numpy()] == r].mean() for r in range(3)]
        rows.append({"Regime source": name, "Argmax agreement w/ HMM": round(float((am == P0.to_numpy().argmax(1)).mean()), 3),
                     "Mean |dP| vs HMM": round(float(np.abs(P.to_numpy() - P0.to_numpy()).sum(1).mean() / 2), 3),
                     "Switch rate (stability)": round(float((np.diff(am) != 0).mean()), 3),
                     "Spearman(E[regime], fwd vol)": round(float(spearmanr(e[ok.to_numpy()], f[ok])[0]), 3),
                     "Fwd vol | Low/Med/High": " / ".join(f"{b:.4f}" for b in by)})
    return pd.DataFrame(rows).set_index("Regime source")


In [ ]:
# cross-check the fast simulator against a Qiskit circuit
rng = np.random.default_rng(0); th_t = init_params(rng); Xt = np.clip(rng.normal(size=(4, 3)), -3, 3)
print("VQC numpy vs Qiskit, max |diff| in <Z>:", np.abs(vqc_expZ(th_t, Xt) - np.array([vqc_qiskit_expZ(th_t, x) for x in Xt])).max())

t0 = time.time(); _VQC_STATE["th"] = None
panels_vqc = build_panels(vqc_panel); print(f"VQC panels built in {time.time()-t0:.0f}s")
named = {"HMM (classical)": panels_hmm, "VQC (quantum)": panels_vqc, "Threshold rule (non-quantum)": panels_thr}
tab7_reg = regime_report(named); display(tab7_reg)

rs7 = [run_backtest(pn, solver_brute, lab) for lab, pn in
       [("Regimes: HMM (brute force)", panels_hmm), ("Regimes: VQC (brute force)", panels_vqc), ("Regimes: threshold rule (brute force)", panels_thr)]]
tab7_fin = summarize(rs7, bench); display(tab7_fin)

fig, axs = plt.subplots(3, 1, figsize=(11, 6), sharex=True)
for ax, (name, pan) in zip(axs, named.items()):
    P = oos_probs(pan); ax.plot(P.index, P.to_numpy() @ np.arange(3)); ax.set_ylabel("E[regime]"); ax.set_title(name, fontsize=9); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 8. Save results

In [ ]:
import os, json
os.makedirs("results", exist_ok=True)
tables = {"phase3_classical": tab3, "phase5_rq2_summary": tab5, "phase5_circuit_resources": tab5_res, "phase6_optimizer": tab6_diag,
          "phase6_financial": tab6_fin, "phase7_regime_quality": tab7_reg, "phase7_financial": tab7_fin}
for k, df in tables.items(): df.to_csv(f"results/{k}.csv")
rq2.to_csv("results/phase5_rq2_raw.csv", index=False)
allres = [bt_bf, bt_sa] + qaoa_res + rs7 + [bench]
pd.DataFrame({r["label"]: r["daily"] for r in allres}).to_csv("results/daily_returns_all_strategies.csv")
json.dump(dict(stocks=stocks, K=K, KAPPA=KAPPA, LAMBDA_R=LAMBDA_R, MIN_TRAIN=MIN_TRAIN, REFIT_EVERY=REFIT_EVERY, REBAL_EVERY=REBAL_EVERY,
               COST_BPS=COST_BPS, ALPHA=ALPHA, P_DEPTH=P_DEPTH, TARGET=TARGET, qiskit=qiskit.__version__),
          open("results/config_used.json", "w"), indent=2)
print("saved:", sorted(os.listdir("results")))